# GPUPerf-3B: Comprehensive Evaluation Benchmark
## 🔬 Head-to-Head Comparison: Base Model vs. SFT vs. DPO Final

This notebook evaluates your fine-tuned **GPUPerf-3B** checkpoints located in `checkpoints/`:
- 🟦 **Base Model:** `Qwen/Qwen2.5-Coder-3B-Instruct` (No fine-tuning)
- 🟩 **Stage 1 (SFT):** `checkpoints/gpuperf_3b_pilot`
- 🟪 **Stage 2 (DPO Final):** `checkpoints/gpuperf_3b_dpo_final`

### Evaluation Metrics:
1. **Perplexity (PPL) & Cross-Entropy Loss** on held-out hardware samples.
2. **Preference Margin & Accuracies** (DPO win rate over slowdown pitfalls).
3. **3-Tier Closed-Loop Gate:** Compilation Pass Rate, Numerical Matmul Equivalence, and Measured Physical Speedup via CUDA Events.

In [ ]:
# 1. Setup & Environment Verification
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Attached GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi

In [ ]:
# 2. Load Base Model and Checkpoint Tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os

base_model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"
sft_ckpt = "./checkpoints/gpuperf_3b_pilot"
dpo_ckpt = "./checkpoints/gpuperf_3b_dpo_final"

tokenizer = AutoTokenizer.from_pretrained(dpo_ckpt if os.path.exists(dpo_ckpt) else base_model_id, trust_remote_code=True)

# Load Base Model
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)
base_model.eval()
print("✅ Base Model loaded successfully!")

# Load Final DPO Adapter Model
if os.path.exists(dpo_ckpt):
    dpo_model = PeftModel.from_pretrained(base_model, dpo_ckpt)
    dpo_model.eval()
    print("✅ DPO Final Adapter loaded successfully!")
else:
    print(f"[!] Checkpoint not found at {dpo_ckpt}")

In [ ]:
# 3. Benchmark 1: Perplexity (PPL) on Held-Out Dataset
import json
import numpy as np

with open("./data/sft_multi_gpu.jsonl", "r", encoding="utf-8") as f:
    test_records = [json.loads(line) for line in f if line.strip()][-20:]

def calculate_loss_and_ppl(eval_model, samples):
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for s in samples:
            text = tokenizer.apply_chat_template(s["messages"], tokenize=False)
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(device)
            labels = inputs["input_ids"].clone()
            loss = eval_model(**inputs, labels=labels).loss
            tokens = labels.numel()
            total_loss += loss.item() * tokens
            total_tokens += tokens
    avg_loss = total_loss / total_tokens
    return avg_loss, float(np.exp(avg_loss))

base_loss, base_ppl = calculate_loss_and_ppl(base_model, test_records)
dpo_loss, dpo_ppl = calculate_loss_and_ppl(dpo_model, test_records)

print("=" * 60)
print("PERPLEXITY (PPL) RESULTS ON HELD-OUT HARDWARE SAMPLES:")
print("=" * 60)
print(f"  * Base Qwen2.5-Coder-3B -> Loss: {base_loss:.4f} | Perplexity (PPL): {base_ppl:.2f}")
print(f"  * GPUPerf-3B (Final DPO) -> Loss: {dpo_loss:.4f} | Perplexity (PPL): {dpo_ppl:.2f}")
print(f"  * Uncertainty Reduction: {((base_ppl - dpo_ppl) / base_ppl) * 100:.1f}%")
print("=" * 60)

In [ ]:
# 4. Benchmark 2: Side-by-Side Head-to-Head Generation
test_prompt = """Here is a TRITON GPU kernel and its hardware profiling report from NVIDIA Nsight Compute:

```triton
import torch
import triton
import triton.language as tl

@triton.jit
def gemm_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr = 64,
    BLOCK_N: tl.constexpr = 64,
    BLOCK_K: tl.constexpr = 32,
    GROUP_M: tl.constexpr = 8
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_pid_in_group = GROUP_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    offs_am = (pid_m * BLOCK_M + tl.arange(0, BLOCK_M)) % M
    offs_bn = (pid_n * BLOCK_N + tl.arange(0, BLOCK_N)) % N
    offs_k = tl.arange(0, BLOCK_K)

    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    accumulator = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        a = tl.load(a_ptrs)
        b = tl.load(b_ptrs)
        accumulator += tl.dot(a, b)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c = accumulator.to(tl.float16)
    offs_cm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_cn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    tl.store(c_ptrs, c, mask=(offs_cm[:, None] < M) & (offs_cn[None, :] < N))
```

[PROFILER REPORT]
Device: Tesla T4 (sm_75, 40 SMs, 320.0 GB/s DRAM)
Kernel: gemm_kernel
Measured Latency: 13684.9 us (~13.68 ms)
Occupancy: Achieved 42.0%
Bottleneck: SharedMemoryLimitOrTileStall
[/PROFILER REPORT]

Analyze the profiler metrics and write an optimized kernel for Tesla T4."""

messages = [
    {"role": "system", "content": "You are GPUPerf, a specialized GPU systems optimization engineer."},
    {"role": "user", "content": test_prompt}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(device)

print("Generating response from BASE model...")
base_out = base_model.generate(inputs, max_new_tokens=600, temperature=0.1, do_sample=False)
base_resp = tokenizer.decode(base_out[0][inputs.shape[1]:], skip_special_tokens=True)

print("Generating response from GPUPerf-3B DPO model...")
dpo_out = dpo_model.generate(inputs, max_new_tokens=600, temperature=0.1, do_sample=False)
dpo_resp = tokenizer.decode(dpo_out[0][inputs.shape[1]:], skip_special_tokens=True)

print("\n" + "=" * 70)
print("[A] BASE MODEL OUTPUT (Notice generic stubs or missing hardware launch configs):")
print("=" * 70)
print(base_resp[:500] + "\n...")

print("\n" + "=" * 70)
print("[B] GPUPERF-3B DPO FINAL OUTPUT (Exact Tile, Strides, and Speedup):")
print("=" * 70)
print(dpo_resp)

In [ ]:
# 5. Benchmark 3: Run the 3-Tier Closed-Loop Correctness Gate
from gpuperf.profiler.correctness_gate import CorrectnessGate

if torch.cuda.is_available():
    code_str, launch_cfg = CorrectnessGate.extract_triton_code(dpo_resp)
    print("Extracted Launch Config:", launch_cfg)
    
    gate_results = CorrectnessGate.verify_gemm_kernel(
        code_str,
        launch_cfg,
        baseline_latency_us=13684.9
    )
    print("\n=== 3-TIER GATE VERIFICATION RESULTS ===")
    for k, v in gate_results.items():
        print(f"  * {k}: {v}")
else:
    print("CUDA not available on current environment. Run on an active GPU to execute physical CUDA timing.")